# 02. Sentiment Classification and Validation

This notebook documents the final dissertation modeling design using a fine-tuned `distilbert-base-uncased` classifier and a TF-IDF + SVM baseline.

The evaluation focuses on whether the model produces reliable information for future reporting, not only whether it performs well on a random test sample.

> The private institutional dataset is required to reproduce the reported metrics and is intentionally excluded from this public repository.


## Evaluation design

- **Random split:** 80% train, 10% validation, 10% test, stratified by sentiment
- **Temporal split:** 2022/23 and 2023/24 for training/validation; 2024/25 held out as unseen future data
- **Main model:** fine-tuned DistilBERT
- **Baseline:** TF-IDF + linear SVM
- **Epochs:** 3
- **Batch size:** 16
- **Learning rate:** 2e-5
- **Optimizer:** AdamW
- **Metrics:** accuracy, precision, recall, F1, confusion matrix

The baseline shows whether the contextual model adds meaningful classification value, while the temporal split tests how well the approach transfers to feedback collected in a later academic year.


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = 'distilbert-base-uncased'
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Load the private labeled dataset

The original dataset is intentionally excluded from GitHub. To reproduce the study, provide a private CSV with these columns: `academic_year`, `comment_preprocessed` (or `comment`), and `sentiment`.

In [ ]:
# Example only: replace with your private local path when reproducing the dissertation
PRIVATE_DATA_PATH = '../data/private/cleaned_labeled_feedback.csv'

# df = pd.read_csv(PRIVATE_DATA_PATH)
# text_col = 'comment_preprocessed' if 'comment_preprocessed' in df.columns else 'comment'
# df['label'] = df['sentiment'].str.lower().map(LABEL_MAP)
# df = df.dropna(subset=[text_col, 'label', 'academic_year']).copy()

## Scenario 1: random split

This scenario provides a controlled model comparison under a standard stratified split. It is useful for benchmarking, but it does not fully represent how the model would be used on future feedback.


In [ ]:
def random_80_10_10_split(df, text_col):
    train_df, temp_df = train_test_split(
        df, test_size=0.20, random_state=SEED, stratify=df['label']
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label']
    )
    return train_df, val_df, test_df

## Scenario 2: temporal out-of-sample validation

Historical academic years are used to train the model and the latest year is held out. This more closely reflects a real reporting process in which past feedback is used to classify newly collected responses.


In [ ]:
def temporal_split(df):
    historical = df[df['academic_year'].isin(['2022/23', '2023/24'])].copy()
    test_df = df[df['academic_year'].eq('2024/25')].copy()

    train_df, val_df = train_test_split(
        historical,
        test_size=0.10,
        random_state=SEED,
        stratify=historical['label']
    )
    return train_df, val_df, test_df

## DistilBERT dataset and model setup

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, num_labels=3, id2label=ID_TO_LABEL, label2id=LABEL_MAP
# ).to(device)

## Baseline: TF-IDF + SVM

The baseline provides a reference point for assessing whether the contextual transformer model improves classification enough to justify the added complexity.


In [ ]:
def train_svm_baseline(train_texts, train_labels, test_texts):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    model = LinearSVC(random_state=SEED)
    model.fit(X_train, train_labels)
    return model.predict(X_test)

## Reported dissertation results

These are fixed results reported in the final dissertation and are not recomputed from the synthetic public sample. They are included here to show the model comparison and the change in performance across evaluation scenarios.


In [ ]:
reported_results = pd.DataFrame([
    ['Random', 'DistilBERT', 0.81, 0.92, 0.63, 0.63],
    ['Random', 'SVM',        0.72, 0.80, 0.62, 0.59],
    ['Temporal', 'DistilBERT', 0.87, 0.93, 0.84, 0.55],
], columns=['Evaluation', 'Model', 'Accuracy', 'Positive F1', 'Negative F1', 'Neutral F1'])
reported_results

## Interpretation

DistilBERT outperformed the SVM baseline on the random split. On the temporal holdout, overall accuracy and negative-class F1 improved, while neutral-class F1 weakened.

This means the later-period model was stronger at identifying negative feedback, but neutral and suggestion-oriented comments remained more ambiguous. If the classification were used to support reporting or prioritization, those cases would still benefit from human review rather than full automation.

The comparison therefore highlights both the value of the model and the specific area where its output should be interpreted with more caution.
